# Five-Factor Equity Screener — Research Notebook

**Author:** Mike | **Framework version:** v0.4 | **Date of run:** rendered inline below

## The problem

Retail investors face ~40,000+ investable equities globally. Discretionary bottom-up analysis
of each is impossible. A systematic filter is required to shrink the universe to a
tractable candidate set for deeper qualitative work.

## The framework

This screener applies a classical five-factor value/quality filter over the US + Europe
universe (~1,500 tickers after cleaning), then ranks survivors by a z-scored composite:

| Factor | Rationale | Direction |
|---|---|---|
| **P/E (TTM)** | Valuation — how much am I paying per unit of earnings? | Lower is better |
| **ROE (FQ)** | Quality — how efficiently is management deploying equity? | Higher is better |
| **Debt/Equity** | Balance-sheet risk | Lower is better |
| **PEG (TTM)** | Growth-adjusted valuation | Lower is better |
| **Free Cash Flow (TTM)** | Cash generation & moat sustainability | Positive required |

Data source: TradingView official screener API (via `tradingview-screener` v3.2.1).

## Design decisions worth flagging upfront

- **Financials excluded.** Bank FCF and D/E are not comparable to industrials; they require
  a separate framework (P/TBV, CET1, ROTE). Standard practice in equity research
  (Damodaran, McKinsey).
- **Only primary listings.** A ticker passes only if its exchange is a domestic venue for
  its country of incorporation. Eliminates depositary receipts, cross-listings, and
  duplicate share classes.
- **Micro-caps excluded** ($1B market-cap floor). Below that, TradingView's fundamentals
  data quality degrades.
- **Hard filters + ranking**, not pure ranking. Any ticker failing any threshold is
  dropped before scoring. The composite is used only to *order* survivors, not to
  compensate for a failed factor.

## Disclaimer

Educational research only. Not investment advice. Screener output is a starting point for
further fundamental due diligence, not a buy/sell signal.

---

## 1 · Live fetch and universe construction

The next cell hits the TradingView API in real time. Expect 30–60 seconds while
8 markets are fetched (US, Germany, France, Italy, Spain, Netherlands, Switzerland, UK).

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

# Make the screener importable
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from run_screener import run
from src import config

# Aesthetics
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Config profile in use:")
print(f"  PE_MAX  = {config.PE_MAX}")
print(f"  ROE_MIN = {config.ROE_MIN}%")
print(f"  DE_MAX  = {config.DE_MAX}")
print(f"  PEG_MAX = {config.PEG_MAX}")
print(f"  FCF_MIN = {config.FCF_MIN}")

In [ ]:
# Live fetch — takes 30-60 seconds
raw, clean, ranked, quarantine = run(verbose=True)

---

## 2 · Universe — what's in and what's out

Before trusting any ranking, we should understand the shape of the pool that survived
the cleaning stage. Two things to verify visually:

1. **Geographic balance:** the fetch shouldn't be dominated by one country.
2. **Sector balance:** the cleaning shouldn't have accidentally wiped out a whole sector.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Country breakdown
country_counts = clean['country'].value_counts().head(10)
axes[0].barh(country_counts.index[::-1], country_counts.values[::-1], color='steelblue')
axes[0].set_title(f'Cleaned universe — top 10 countries ({len(clean)} tickers)', fontsize=11)
axes[0].set_xlabel('Number of tickers')

# Sector breakdown
sector_counts = clean['sector'].value_counts()
axes[1].barh(sector_counts.index[::-1], sector_counts.values[::-1], color='steelblue')
axes[1].set_title('Cleaned universe — sectors', fontsize=11)
axes[1].set_xlabel('Number of tickers')

plt.tight_layout()
plt.show()

print(f"\nGeographic split (top 5): ")
print((country_counts.head(5) / len(clean) * 100).round(1).astype(str) + '%')

In [ ]:
# Market cap distribution — log scale to see the whole range
fig, ax = plt.subplots(figsize=(10, 4))
mcap_bn = clean['market_cap_basic'] / 1e9

ax.hist(np.log10(mcap_bn), bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('log10(Market cap in $B)')
ax.set_ylabel('Number of tickers')
ax.set_title(f'Market cap distribution — cleaned universe (n={len(clean)})')

# Add reference lines
for label, val in [('$1B', 1), ('$10B', 10), ('$100B', 100), ('$1T', 1000)]:
    ax.axvline(np.log10(val), color='grey', linestyle='--', alpha=0.5)
    ax.text(np.log10(val), ax.get_ylim()[1]*0.9, label, ha='center',
            fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

print(f"Median market cap: ${mcap_bn.median():.2f}B")
print(f"Largest company:   {clean.iloc[0]['description']} (${mcap_bn.max():.0f}B)")

---

## 3 · The filter funnel — where does the universe die?

Each of the five factors kills a chunk of the universe. Visualizing *where* each factor
cuts tells us whether our thresholds are calibrated. If a factor kills 90% of survivors,
either the threshold is too tight, or that factor is genuinely rare (which itself is
a research finding).

In [ ]:
from src.screener import FACTOR_COLS

# Universe after drop-NaN and sanity bounds — the pool that faces the five-factor test
required = list(FACTOR_COLS.values())
sanity_bounds_ok = clean.dropna(subset=required).copy()
for c, (lo, hi) in config.SANITY_BOUNDS.items():
    if c in sanity_bounds_ok.columns:
        sanity_bounds_ok = sanity_bounds_ok[
            (sanity_bounds_ok[c] >= lo) & (sanity_bounds_ok[c] <= hi)
        ]

# Individual pass rates per factor (each in isolation)
individual = {
    'P/E < ' + str(config.PE_MAX):     (sanity_bounds_ok['price_earnings_ttm'] > 0) &
                                        (sanity_bounds_ok['price_earnings_ttm'] < config.PE_MAX),
    'ROE > ' + str(config.ROE_MIN) + '%': sanity_bounds_ok['return_on_equity_fq'] > config.ROE_MIN,
    'D/E < ' + str(config.DE_MAX):     sanity_bounds_ok['debt_to_equity'] < config.DE_MAX,
    'PEG < ' + str(config.PEG_MAX):    (sanity_bounds_ok['price_earnings_growth_ttm'] > 0.01) &
                                        (sanity_bounds_ok['price_earnings_growth_ttm'] < config.PEG_MAX),
    'FCF > 0':                         sanity_bounds_ok['free_cash_flow_ttm'] > config.FCF_MIN,
}

pass_rates = pd.DataFrame({
    'Factor': list(individual.keys()),
    'Passes (n)': [m.sum() for m in individual.values()],
    'Pass rate': [f"{m.mean()*100:.1f}%" for m in individual.values()],
})
print("Individual factor pass rates (each in isolation):")
print(pass_rates.to_string(index=False))
print(f"\nAll five combined (AND): {len(ranked)} survivors ({len(ranked)/len(sanity_bounds_ok)*100:.1f}% of pool)")

In [ ]:
# Funnel chart — where the universe dies stage by stage
stages = ['Raw fetch', 'Cleaned', 'Non-NaN + sanity', 'Five-factor survivors']
counts = [len(raw), len(clean), len(sanity_bounds_ok), len(ranked)]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(stages[::-1], counts[::-1], color=['#c72e29', '#e59f42', '#6ba4d3', '#3d5a80'])
ax.set_xlabel('Number of tickers')
ax.set_title('Screener funnel')
for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + max(counts)*0.01, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=10)
ax.set_xlim(0, max(counts) * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of each factor with threshold line
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

factor_specs = [
    ('price_earnings_ttm', config.PE_MAX, 'P/E (TTM)', 'less'),
    ('return_on_equity_fq', config.ROE_MIN, 'ROE % (FQ)', 'greater'),
    ('debt_to_equity', config.DE_MAX, 'Debt / Equity', 'less'),
    ('price_earnings_growth_ttm', config.PEG_MAX, 'PEG (TTM)', 'less'),
    ('free_cash_flow_ttm', config.FCF_MIN, 'Free Cash Flow ($B TTM)', 'greater'),
]

for ax, (col_name, threshold, label, direction) in zip(axes, factor_specs):
    values = sanity_bounds_ok[col_name].dropna()
    if col_name == 'free_cash_flow_ttm':
        values = values / 1e9
        threshold_plot = threshold / 1e9
    else:
        threshold_plot = threshold

    # Clip for display
    q_low, q_high = values.quantile([0.02, 0.98])
    values_clipped = values.clip(q_low, q_high)

    ax.hist(values_clipped, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(threshold_plot, color='crimson', linestyle='--', linewidth=2,
               label=f'threshold = {threshold_plot:.2f}')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('')
    ax.legend(loc='upper right', fontsize=9)

axes[-1].axis('off')
plt.suptitle('Factor distributions — cleaned universe (2%–98% clipped for display)',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

---

## 4 · Survivors — ranking and factor structure

The composite score is a weighted sum of z-scored factors (weights in `config.py`):
`PE 20%, ROE 30%, D/E 15%, PEG 20%, FCF 15%`. Higher z on lower-is-better factors is
sign-inverted, so higher composite = better across the board.

In [ ]:
# Composite score distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ranked['score'], bins=25, color='steelblue', edgecolor='white')
ax.axvline(ranked['score'].median(), color='grey', linestyle='--', label=f'median = {ranked["score"].median():.2f}')
ax.axvline(ranked.iloc[config.TOP_N-1]['score'], color='crimson', linestyle='--',
           label=f'top-{config.TOP_N} cutoff = {ranked.iloc[config.TOP_N-1]["score"]:.2f}')
ax.set_xlabel('Composite score (higher = better)')
ax.set_ylabel('Number of survivors')
ax.set_title(f'Composite score distribution — {len(ranked)} five-factor survivors')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top-N table
top_n = ranked.head(config.TOP_N).copy()
display_cols = {
    'name': 'Ticker',
    'description': 'Company',
    'country': 'Country',
    'sector': 'Sector',
    'price_earnings_ttm': 'P/E',
    'return_on_equity_fq': 'ROE %',
    'debt_to_equity': 'D/E',
    'price_earnings_growth_ttm': 'PEG',
    'free_cash_flow_ttm': 'FCF ($B)',
    'score': 'Score',
}
tbl = top_n[list(display_cols.keys())].rename(columns=display_cols).copy()
tbl['FCF ($B)'] = tbl['FCF ($B)'] / 1e9
tbl.index = range(1, len(tbl) + 1)
tbl.style.format({
    'P/E': '{:.1f}', 'ROE %': '{:.1f}', 'D/E': '{:.2f}',
    'PEG': '{:.2f}', 'FCF ($B)': '{:.2f}', 'Score': '{:+.2f}',
}).background_gradient(subset=['Score'], cmap='RdYlGn')

In [ ]:
# Heatmap of the top-N vs the survivor pool median — how much better are they?
survivor_medians = ranked[list(FACTOR_COLS.values())].median()
top = top_n[list(FACTOR_COLS.values())].copy()

# Ratio to median (for higher-is-better) or inverse ratio (for lower-is-better)
lower_is_better = {'price_earnings_ttm', 'debt_to_equity', 'price_earnings_growth_ttm'}
comparison = top.copy()
for c in top.columns:
    if c in lower_is_better:
        comparison[c] = survivor_medians[c] / top[c]
    else:
        comparison[c] = top[c] / survivor_medians[c]

comparison.index = top_n['name'].values
comparison.columns = ['P/E', 'ROE', 'D/E', 'PEG', 'FCF']

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(comparison, annot=True, fmt='.2f', cmap='RdYlGn', center=1.0,
            cbar_kws={'label': 'Ratio to survivor pool median (higher = better)'},
            linewidths=0.5, ax=ax)
ax.set_title('Top-10 factors relative to survivor pool median')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: two of the most informative factor pairs
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PE vs ROE — the classic value/quality space
axes[0].scatter(ranked['price_earnings_ttm'], ranked['return_on_equity_fq'],
                c=ranked['score'], cmap='RdYlGn', s=40, alpha=0.7, edgecolor='k', linewidth=0.3)
axes[0].scatter(top_n['price_earnings_ttm'], top_n['return_on_equity_fq'],
                s=120, facecolors='none', edgecolors='black', linewidths=1.5, label=f'Top {config.TOP_N}')
for _, row in top_n.iterrows():
    axes[0].annotate(row['name'], (row['price_earnings_ttm'], row['return_on_equity_fq']),
                     xytext=(4, 4), textcoords='offset points', fontsize=8)
axes[0].set_xlabel('P/E (TTM)')
axes[0].set_ylabel('ROE % (FQ)')
axes[0].set_title('Value/Quality space')
axes[0].legend(loc='upper right')

# PEG vs FCF
axes[1].scatter(ranked['price_earnings_growth_ttm'], ranked['free_cash_flow_ttm']/1e9,
                c=ranked['score'], cmap='RdYlGn', s=40, alpha=0.7, edgecolor='k', linewidth=0.3)
axes[1].scatter(top_n['price_earnings_growth_ttm'], top_n['free_cash_flow_ttm']/1e9,
                s=120, facecolors='none', edgecolors='black', linewidths=1.5)
for _, row in top_n.iterrows():
    axes[1].annotate(row['name'],
                     (row['price_earnings_growth_ttm'], row['free_cash_flow_ttm']/1e9),
                     xytext=(4, 4), textcoords='offset points', fontsize=8)
axes[1].set_xlabel('PEG (TTM)')
axes[1].set_ylabel('FCF (TTM, $B)')
axes[1].set_yscale('symlog')
axes[1].set_title('Growth/Cash-generation space')

plt.tight_layout()
plt.show()

---

## 5 · Top-10 breakdown — geography and sector

Are the winners concentrated in one region or sector? A healthy screener should produce
some diversification even without imposing it; if not, that's a signal that the
current threshold profile has a hidden bias.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

top_n['country'].value_counts().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title(f'Top-{config.TOP_N} by country')
axes[0].set_xlabel('Number of tickers')

top_n['sector'].value_counts().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title(f'Top-{config.TOP_N} by sector')
axes[1].set_xlabel('Number of tickers')

plt.tight_layout()
plt.show()

---

## 6 · Stock pitch — the chosen candidate

*The screener has done its job: from ~8,500 fetched tickers to a ranked short-list of
survivors. The final selection for deep-dive is a qualitative judgement call — the
composite score is one input, not the answer.*

**Decision framework:** among the top-10, prefer candidates where:

- The tesis is understandable in one sentence
- Data are not obviously distorted by one-off items (buyback-driven ROE, cyclical peak margins)
- Position size fits the portfolio (currently ~5–8% for tactical trades)
- Catalyst or re-rating driver is visible on a 12–24 month horizon

Fill in the fields below after reviewing the evidence above.

### Candidate: _[ticker]_

**Company:** _[full name]_

**Country / Sector:** _[…]_

**Five-factor snapshot:** P/E _[…]_ · ROE _[…]%_ · D/E _[…]_ · PEG _[…]_ · FCF $_[…]B_

---

**One-line thesis**

_[Fill in]_

**Why it screens well (quantitative)**

- _[…]_
- _[…]_
- _[…]_

**Qualitative diligence — beyond the numbers**

- Business model / moat: _[…]_
- Management / capital allocation: _[…]_
- Key risks: _[…]_
- Catalyst / expected re-rating driver (12–24m): _[…]_

**Entry / risk plan**

- Target sizing: _[X% of portfolio]_
- Entry level: _[$X or opportunistic]_
- Mental stop: _[$X — invalidation of the thesis]_
- Time horizon: _[…]_

**Post-mortem hook**

_Once the position is closed, revisit this section with realized IRR vs thesis and
what to change in the framework for the next candidate._

---

## Appendix · Reproducibility

Full ranking (all survivors) available as Excel via:

```python
from src import report
report.export_excel(ranked, quarantine=quarantine)
```

The screener's configuration (thresholds, weights, universe) is fully declarative in
`src/config.py`. All thresholds can be tuned there without touching the pipeline logic.

**References:**

- Damodaran, A. *Investment Valuation* (3rd ed.)
- Piotroski, J. (2000) *Value Investing: Historical Financial Statement Information*
- Greenblatt, J. *The Little Book That Still Beats the Market*